# Auto-label Village plate photos (Grounding DINO + SAM 2.1)
Runtime: T4 GPU. Photos live in `MyDrive/leftover-photos.zip` (a flat zip of JPEGs), named `item_plate#_state_##.jpg`
(e.g. `fries_p2_scraps_03.jpg`, `empty_p1_clean_01.jpg`; mixed plates join slugs with `+`: `asparagus+chicken_p1_half_04.jpg`). Item slugs are in `labeling/pilot_items.yaml`.
**Always look at the overlay grid before training**: auto-labels are a draft, not ground truth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/leftover-lens && mkdir -p /content && unzip -q -o /content/drive/MyDrive/leftover-lens.zip -d /content
%cd /content/leftover-lens
!pip install -q -U transformers pillow-heif opencv-python-headless pyyaml
!pip install -q -e . --no-deps

In [ ]:
import cv2, numpy as np, pillow_heif
from pathlib import Path
from PIL import Image, ImageOps
pillow_heif.register_heif_opener()

!rm -rf /content/leftover-photos && mkdir -p /content/leftover-photos && unzip -q -o /content/drive/MyDrive/leftover-photos.zip -d /content/leftover-photos
PHOTOS = Path('/content/leftover-photos')
WORK = Path('/content/photos_jpg'); WORK.mkdir(exist_ok=True)

# Normalise everything (HEIC/JPG/PNG) to upright JPEGs, longest side 1600 px.
paths = sorted(p for p in PHOTOS.rglob('*') if p.suffix.lower() in {'.heic', '.jpg', '.jpeg', '.png'})
for p in paths:
    im = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
    im.thumbnail((1600, 1600))
    im.save(WORK / (p.stem + '.jpg'), quality=95)
print(len(paths), 'photos')

In [ ]:
from labeling.autolabel import Autolabeler, load_pilot_items, missing_items, parse_photo_name, split_items
items = load_pilot_items()
labeler = Autolabeler()          # downloads Grounding DINO base + SAM 2.1 small
results = {}
for p in sorted(WORK.glob('*.jpg')):
    item, _ = parse_photo_name(p)
    if any(s != 'empty' and s not in items for s in split_items(item)):
        print('SKIP unknown item slug:', p.name); continue
    img = cv2.imread(str(p))
    results[p.name] = (img, labeler.label(img, item, items))
    miss = missing_items(item, results[p.name][1], items)
    tag = f'  MISSING {miss}' if miss else ''
    print(f'{p.name:45s}', sorted({m.class_name for m in results[p.name][1]}), tag)
incomplete = {n for n in results if missing_items(parse_photo_name(n)[0], results[n][1], items)}
print(f'\n{len(results) - len(incomplete)} complete, {len(incomplete)} incomplete (auto-excluded):')
print(sorted(incomplete))

In [ ]:
# Review grid: every photo with its masks. Look for: plate missing, food mask = whole plate,
# napkin/fork labelled as food, only part of the food covered.
import matplotlib.pyplot as plt
from core.types import SEG_CLASSES
rng = np.random.RandomState(1)
colors = {c: rng.randint(60, 255, 3) for c in SEG_CLASSES}
names = list(results)
cols = 4; rows = (len(names) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows), squeeze=False)
for ax in axes.ravel(): ax.axis('off')
for ax, n in zip(axes.ravel(), names):
    img, labels = results[n]
    over = img.copy()
    for m in labels:
        over[m.mask] = (0.5 * over[m.mask] + 0.5 * colors[m.class_name]).astype(np.uint8)
    ax.imshow(over[:, :, ::-1]); ax.set_title(('[MISSING] ' if n in incomplete else '') + n + '\n' + ', '.join(m.class_name for m in labels), fontsize=8)
plt.tight_layout(); plt.show()

## Drop bad labels
Photos whose foods were not all found are already excluded (`incomplete`). Add any other photo whose masks look wrong in the grid (napkin or fork labelled as food, a mask that covers the whole plate).


In [ ]:
import yaml

# Hand exclusions from the review grid live in labeling/exclusions.yaml (add new ones there,
# with the reason, so the dataset can be rebuilt exactly).
with open("labeling/exclusions.yaml") as f:
    EXCLUDE = set(yaml.safe_load(f)["manual"])
EXCLUDE |= incomplete
print(len(EXCLUDE), "excluded,", len(results) - len(EXCLUDE), "kept")


In [ ]:
import shutil, yaml
from labeling.autolabel import split_groups, to_yolo_seg_lines
OUT = Path('/content/datasets/village'); shutil.rmtree(OUT, ignore_errors=True)
good = [n for n in results if n not in EXCLUDE and results[n][1]]
val_groups = split_groups([parse_photo_name(n)[1] for n in good])
counts = {'train': 0, 'val': 0}
for n in good:
    split = 'val' if parse_photo_name(n)[1] in val_groups else 'train'
    (OUT / 'images' / split).mkdir(parents=True, exist_ok=True); (OUT / 'labels' / split).mkdir(parents=True, exist_ok=True)
    img, labels = results[n]
    cv2.imwrite(str(OUT / 'images' / split / n), img)
    (OUT / 'labels' / split / (Path(n).stem + '.txt')).write_text('\n'.join(to_yolo_seg_lines(labels)))
    counts[split] += 1
(OUT / 'data.yaml').write_text(yaml.safe_dump({'path': str(OUT), 'train': 'images/train', 'val': 'images/val', 'names': dict(enumerate(SEG_CLASSES))}))
print(counts, 'held-out groups:', sorted(val_groups))
shutil.make_archive('/content/drive/MyDrive/village-dataset', 'zip', OUT)
print('saved MyDrive/village-dataset.zip')